# Publication-safe notebook

This generalized reference implementation expects a live SparkSession named
`spark` and pseudonymous tables matching `docs/data-contract.md`. It contains no
institution-specific ingestion, credentials, internal paths, data, or results.

Parameter defaults are illustrative and intentionally differ from internal
operating settings. Calibrate them only with authorized development data.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark import StorageLevel
import numpy as np
import pandas as pd

import os

WRITE_DB = os.environ.get("FRAUD_GRAPH_WRITE_DB", "fraud_graph_demo")
EMP = os.environ.get("FRAUD_GRAPH_PREFIX", "onboarding_fraud")
EDGES_TXN  = f"{WRITE_DB}.{EMP}_v2_edges_transferred"
EDGES_ATM  = f"{WRITE_DB}.{EMP}_v2_edges_withdrew"
SUBG_NIDX  = f"{WRITE_DB}.{EMP}_v2_subgraph_node_index"
OWN_TBL    = f"{WRITE_DB}.{EMP}_v2_edges_ownership"
FEATS_CUST = f"{WRITE_DB}.{EMP}_v2_cust_features"

ANCHOR_TRAILING = {
    "202401": ["202310", "202311", "202312"],
    "202402": ["202311", "202312", "202401"],
    "202403": ["202312", "202401", "202402"],
}

TEST_ONE_ANCHOR = False
USE_GRAPHFRAMES = True
RARE_BENEF_MAX = int(os.environ.get("FRAUD_GRAPH_RARE_BENEF_MAX", "25"))
RARE_ATM_MAX = int(os.environ.get("FRAUD_GRAPH_RARE_ATM_MAX", "250"))

K_PCTS = [float(value) for value in os.environ.get("FRAUD_GRAPH_EVALUATION_PCTS", "0.02,0.05").split(",")]

RUN_REAL_FRAUDAR   = True
FRAUDAR_EPS = float(os.environ.get("FRAUD_GRAPH_FRAUDAR_EPSILON", "0.5"))
FRAUDAR_MAX_ROUNDS = int(os.environ.get("FRAUD_GRAPH_FRAUDAR_MAX_ROUNDS", "15"))
FRAUDAR_MAX_BLOCKS = int(os.environ.get("FRAUD_GRAPH_FRAUDAR_MAX_BLOCKS", "5"))
FRAUDAR_MIN_ACCT = int(os.environ.get("FRAUD_GRAPH_FRAUDAR_MIN_ACCOUNTS", "2"))
FRAUDAR_MAX_ACCT = int(os.environ.get("FRAUD_GRAPH_FRAUDAR_MAX_ACCOUNTS", "2500"))

OUT_MOTIF           = f"{WRITE_DB}.{EMP}_v2_motif_features"
OUT_MOTIF_CUST      = f"{WRITE_DB}.{EMP}_v2_motif_cust_features"
OUT_MOTIF_CUST_FULL = f"{WRITE_DB}.{EMP}_v2_motif_cust_full"
OUT_MOTIF_LIFT      = f"{WRITE_DB}.{EMP}_v2_motif_increment_lift"
OUT_MOTIF_SLOPE     = f"{WRITE_DB}.{EMP}_v2_motif_sloping"

spark.conf.set("spark.sql.shuffle.partitions", os.environ.get("FRAUD_GRAPH_SHUFFLE_PARTITIONS", "200"))

def save_overwrite(df, t):
    spark.sql("DROP TABLE IF EXISTS " + t); df.write.mode("overwrite").saveAsTable(t)

anchors = (["202401"] if TEST_ONE_ANCHOR else list(ANCHOR_TRAILING.keys()))
print("anchors:", anchors, "| rare benef <=", RARE_BENEF_MAX, "| rare ATM <=", RARE_ATM_MAX,
      "| GraphFrames:", USE_GRAPHFRAMES, "| shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))


In [ ]:
GF_JAR = os.environ.get("GRAPHFRAMES_JAR", "graphframes.jar")
CKPT_DIR = os.environ.get("FRAUD_GRAPH_CHECKPOINT_DIR", "file:///tmp/onboarding_fraud/gf_ckpt")
GraphFrame = None
spark.sparkContext.setCheckpointDir(CKPT_DIR)
if USE_GRAPHFRAMES:
    try:
        spark.sparkContext.addPyFile(GF_JAR)
        from graphframes import GraphFrame
        print("GraphFrames enabled (find() motif DSL available).")
    except Exception as ge:
        GraphFrame = None
        print("GraphFrames import failed — 3-cycle motif will use the plain-join fallback:", ge)


In [ ]:
def build_substrate(anchor):
    months = ANCHOR_TRAILING[anchor]
    e0 = (spark.table(EDGES_TXN).where(F.col("txn_month").isin(*months))
          .select("src", "dst", F.col("sum_amount").cast("double").alias("amt"))
          .where(F.col("src") != F.col("dst")).persist(StorageLevel.DISK_ONLY))

    fanin = e0.groupBy("dst").agg(F.countDistinct("src").alias("fi"))
    rare  = fanin.where(F.col("fi") <= RARE_BENEF_MAX).select("dst", "fi").persist(StorageLevel.DISK_ONLY)
    e = (e0.join(rare.select("dst"), "dst", "inner")
         .groupBy("src", "dst").agg(F.sum("amt").alias("amt"))
         .select("src", "dst", F.log1p(F.col("amt")).alias("weight")).persist(StorageLevel.DISK_ONLY))
    e.count(); rare.count(); e0.unpersist()
    lab = spark.table(SUBG_NIDX).where(F.col("anchor_month") == anchor)
    seeds = lab.where(F.col("is_known_bad") == 1).select("id").distinct()
    cands = (lab.where((F.col("is_candidate") == 1) & (F.col("is_known_bad") == 0))
             .select("id", F.col("target_fraud").cast("int").alias("target_fraud")))
    return e, rare, seeds, cands


In [ ]:
def motif_shared_benef(e, rare, seeds):
    seed_benef = (e.join(seeds.select(F.col("id").alias("src")), "src").select("dst").distinct())
    cand_mule  = e.join(seed_benef, "dst").select("src", "dst").distinct()
    base   = cand_mule.groupBy("src").agg(F.countDistinct("dst").alias("n_shared_mule_motif"))
    fanin  = cand_mule.join(rare, "dst").groupBy("src").agg(F.max("fi").alias("max_shared_mule_fanin"))
    cofeed = (e.select(F.col("src").alias("other"), "dst").join(cand_mule.select("src", "dst"), "dst")
              .where(F.col("other") != F.col("src"))
              .groupBy("src").agg(F.countDistinct("other").alias("ring_co_feeders")))
    return (base.join(fanin, "src", "left").join(cofeed, "src", "left")
            .select(F.col("src").alias("id"), "n_shared_mule_motif", "max_shared_mule_fanin", "ring_co_feeders"))


In [ ]:
def motif_atm(anchor, seeds):
    months = ANCHOR_TRAILING[anchor]
    a0 = (spark.table(EDGES_ATM).where(F.col("txn_month").isin(*months))
          .select("src", "dst").where(F.col("src") != F.col("dst")).distinct().persist(StorageLevel.DISK_ONLY))
    afan = a0.groupBy("dst").agg(F.countDistinct("src").alias("afi"))
    rare_atm = afan.where(F.col("afi") <= RARE_ATM_MAX).select("dst")
    a = a0.join(rare_atm, "dst", "inner")
    seed_atm = a.join(seeds.select(F.col("id").alias("src")), "src").select("dst").distinct()
    out = (a.join(seed_atm, "dst").groupBy("src").agg(F.countDistinct("dst").alias("n_shared_atm"))
           .select(F.col("src").alias("id"), "n_shared_atm"))
    a0.unpersist()
    return out


In [ ]:
def _cycles_join(e):
    ab = e.select(F.col("src").alias("a"), F.col("dst").alias("b"))
    bc = e.select(F.col("src").alias("b"), F.col("dst").alias("c"))
    ca = e.select(F.col("src").alias("c"), F.col("dst").alias("a"))
    return ab.join(bc, "b").join(ca, ["c", "a"]).select("a", "b", "c").distinct()

def motif_cycles(e, seeds):
    vertices = (e.select(F.col("src").alias("id")).union(e.select(F.col("dst").alias("id"))).distinct())
    tri = None
    if USE_GRAPHFRAMES and GraphFrame is not None:
        try:
            g = GraphFrame(vertices, e.select("src", "dst"))
            tri = (g.find("(a)-[]->(b); (b)-[]->(c); (c)-[]->(a)")
                   .filter("a.id <> b.id AND b.id <> c.id AND a.id <> c.id")
                   .select(F.col("a.id").alias("a"), F.col("b.id").alias("b"), F.col("c.id").alias("c")))
        except Exception as ce:
            print("   [WARN] GraphFrames find() cycles failed -> join fallback:", ce)
            tri = None
    if tri is None:
        tri = _cycles_join(e)
    tri = tri.persist(StorageLevel.DISK_ONLY); tri.count()

    s = seeds.select("id")
    anch = (tri.join(s.select(F.col("id").alias("a")), "a", "left_semi")
            .unionByName(tri.join(s.select(F.col("id").alias("b")), "b", "left_semi"))
            .unionByName(tri.join(s.select(F.col("id").alias("c")), "c", "left_semi"))
            .select("a", "b", "c").distinct().persist(StorageLevel.DISK_ONLY))
    anch.count(); tri.unpersist()
    on = (anch.select(F.col("a").alias("id")).union(anch.select(F.col("b").alias("id")))
          .union(anch.select(F.col("c").alias("id"))).distinct().withColumn("on_3cycle", F.lit(1)))
    cnt = anch.groupBy(F.col("a").alias("id")).agg(F.count("*").alias("n_3cycle_as_a"))
    out = on.join(cnt, "id", "left").na.fill({"n_3cycle_as_a": 0})
    return out, anch


In [ ]:
def motif_fraudar(e, rare, seeds):

    seed_mules = e.join(seeds.select(F.col("id").alias("src")), "src").select("dst").distinct()
    w   = rare.join(seed_mules, "dst").withColumn("cw", 1.0 / F.log2(F.col("fi") + 5.0)).select("dst", "cw")
    bip = e.join(seed_mules, "dst").select("src", "dst").distinct().join(w, "dst")
    s_acc  = bip.groupBy("src").agg(F.sum("cw").alias("s_acc"))
    d_mule = bip.join(s_acc, "src").groupBy("dst").agg(F.sum("s_acc").alias("d_mule"))
    fr = (bip.join(d_mule, "dst").withColumn("contrib", F.col("cw") * F.col("d_mule"))
          .groupBy("src").agg(F.sum("contrib").alias("fraudar_score")))
    return fr.select(F.col("src").alias("id"), "fraudar_score")


In [ ]:
def _densest_block(bip, eps, max_rounds):

    S_acct = bip.select("src").distinct()
    S_mule = bip.select("dst").distinct()
    best_g = -1.0; best_acct = None; best_mule = None
    for r in range(max_rounds):
        eis = bip.join(S_acct, "src", "left_semi").join(S_mule, "dst", "left_semi")
        f = eis.agg(F.sum("cw").alias("f")).first()["f"]
        if not f or f <= 0:
            break
        ns = S_acct.count() + S_mule.count()
        if ns == 0:
            break
        g = float(f) / ns
        if g > best_g:
            best_g = g
            if best_acct is not None: best_acct.unpersist()
            if best_mule is not None: best_mule.unpersist()
            best_acct = S_acct.persist(StorageLevel.DISK_ONLY); best_acct.count()
            best_mule = S_mule.persist(StorageLevel.DISK_ONLY); best_mule.count()
        thr = 2.0 * (1.0 + eps) * g
        S_acct = eis.groupBy("src").agg(F.sum("cw").alias("d")).where(F.col("d") > thr).select("src").checkpoint(eager=True)
        S_mule = eis.groupBy("dst").agg(F.sum("cw").alias("d")).where(F.col("d") > thr).select("dst").checkpoint(eager=True)
        if S_acct.limit(1).count() == 0 or S_mule.limit(1).count() == 0:
            break
    return best_acct, best_mule, best_g

def motif_fraudar_dense(e, rare, seeds, eps, max_rounds, max_blocks, min_acct, max_acct):

    seed_mules = e.join(seeds.select(F.col("id").alias("src")), "src").select("dst").distinct()
    w = rare.join(seed_mules, "dst").withColumn("cw", 1.0 / F.log2(F.col("fi") + 5.0)).select("dst", "cw")
    bip = e.join(seed_mules, "dst").select("src", "dst").distinct().join(w, "dst").persist(StorageLevel.DISK_ONLY)
    blocks = []
    for blk in range(max_blocks):
        ba, bm, bg = _densest_block(bip, eps, max_rounds)
        if ba is None:
            break
        nacct = ba.count()
        if nacct < min_acct:
            ba.unpersist(); bm.unpersist(); break
        keep = (nacct <= max_acct)
        if keep:
            blocks.append(ba.select(F.col("src").alias("id"))
                          .withColumn("block_id", F.lit(blk)).withColumn("block_density", F.lit(float(bg))))
        print("   fraudar block %d: %d accounts, density=%.4f%s"
              % (blk, nacct, bg, "" if keep else "  (too big -> peeled past, NOT flagged)"))
        bip2 = bip.join(ba, "src", "left_anti").join(bm, "dst", "left_anti").persist(StorageLevel.DISK_ONLY)
        bip2.count(); bip.unpersist(); bip = bip2
        ba.unpersist(); bm.unpersist()
    bip.unpersist()
    if not blocks:
        return None
    allb = blocks[0]
    for bdf in blocks[1:]:
        allb = allb.unionByName(bdf)
    return (allb.groupBy("id").agg(F.max("block_density").alias("block_density"))
            .withColumn("in_dense_block", F.lit(1)))


In [ ]:
def capture_at(df, score_col, ks):
    w = Window.orderBy(F.col(score_col).desc_nulls_last())
    ranked = df.select("target_fraud", F.row_number().over(w).alias("rk"))
    tot = df.agg(F.count("*").alias("n"), F.sum("target_fraud").alias("nbad")).first()
    n, nbad = tot["n"], (tot["nbad"] or 0)
    rows = []
    for p in ks:
        K = max(1, int(n * p))
        topbad = ranked.where((F.col("rk") <= K) & (F.col("target_fraud") == 1)).count()
        cap  = (topbad / nbad) if nbad else 0.0
        lift = ((topbad / K) / (nbad / n)) if nbad else 0.0
        rows.append((p, K, topbad, round(cap, 4), round(lift, 3)))
    return rows, n, nbad

def pr(df, col):
    return df.withColumn("pr_" + col, F.percent_rank().over(Window.orderBy(F.col(col).asc_nulls_first())))

def _woe_iv(cnt, bad):
    good = cnt - bad
    tb, tg = bad.sum(), good.sum()
    pb = (bad / tb).clip(lower=1e-6) if tb else bad * 0 + 1e-6
    pg = (good / tg).clip(lower=1e-6) if tg else good * 0 + 1e-6
    return float(((pg - pb) * np.log(pg / pb)).sum())

def _robust_bins(x, max_bins=10):
    s = pd.Series(np.asarray(x, dtype=float)); nuniq = int(s.nunique(dropna=True))
    if nuniq <= 1: return np.zeros(len(s), dtype=int)
    if nuniq <= max_bins:
        order = {v: i for i, v in enumerate(sorted(s.dropna().unique()))}
        return s.map(order).fillna(0).astype(int).values
    z = (s == 0); out = np.zeros(len(s), dtype=int)
    if z.mean() >= 0.02 and int((~z).sum()) >= max_bins:
        pos = ~z
        qr = pd.qcut(s[pos].rank(method="first"), q=min(max_bins - 1, int((~z).sum())), labels=False, duplicates="drop")
        out[pos.values] = 1 + qr.astype(int).values; return out
    return pd.qcut(s.rank(method="first"), q=max_bins, labels=False, duplicates="drop").astype(int).values


In [ ]:
def motif_passthrough(anchor, cands):
    months = ANCHOR_TRAILING[anchor]
    tx = (spark.table(EDGES_TXN).where(F.col("txn_month").isin(*months))
          .select("src", "dst", F.col("sum_amount").cast("double").alias("amt"))
          .where(F.col("src") != F.col("dst")))
    inflow  = tx.groupBy(F.col("dst").alias("id")).agg(F.sum("amt").alias("in_amt"))
    outflow = tx.groupBy(F.col("src").alias("id")).agg(F.sum("amt").alias("out_amt"))
    return (cands.select("id").join(inflow, "id", "left").join(outflow, "id", "left")
            .na.fill({"in_amt": 0.0, "out_amt": 0.0})
            .withColumn("passthrough_ratio",
                        F.least("in_amt", "out_amt") / (F.greatest("in_amt", "out_amt") + F.lit(1.0)))
            .select("id", "passthrough_ratio"))


In [ ]:
MOTIF_COLS = ["id", "anchor_month", "n_shared_mule_motif", "max_shared_mule_fanin", "ring_co_feeders",
              "n_shared_atm", "on_3cycle", "n_3cycle_as_a", "fraudar_score", "in_dense_block", "block_density",
              "passthrough_ratio"]
first_acc = True; first_cust = True
for a in anchors:
    print("=== motif pass", a, "===")
    e, rare, seeds, cands = build_substrate(a)

    shared = motif_shared_benef(e, rare, seeds)
    atm    = motif_atm(a, seeds)
    fraud  = motif_fraudar(e, rare, seeds)
    try:
        pth = motif_passthrough(a, cands)
    except Exception as pe:
        print("   [WARN] passthrough motif failed -> passthrough_ratio=0:", pe)
        pth = cands.select("id").withColumn("passthrough_ratio", F.lit(0.0))
    try:
        cyc, tri = motif_cycles(e, seeds)
    except Exception as ce:
        print("   [WARN] cycle motif failed -> on_3cycle=0:", ce)
        cyc = cands.select("id").withColumn("on_3cycle", F.lit(0)).withColumn("n_3cycle_as_a", F.lit(0)); tri = None

    dense = None
    if RUN_REAL_FRAUDAR:
        try:
            dense = motif_fraudar_dense(e, rare, seeds, FRAUDAR_EPS, FRAUDAR_MAX_ROUNDS, FRAUDAR_MAX_BLOCKS, FRAUDAR_MIN_ACCT, FRAUDAR_MAX_ACCT)
        except Exception as fe:
            print("   [WARN] distributed Fraudar failed -> in_dense_block=0:", fe); dense = None
    if dense is None:
        dense = cands.select("id").limit(0).withColumn("in_dense_block", F.lit(1)).withColumn("block_density", F.lit(0.0))

    feats = (cands.join(shared, "id", "left").join(atm, "id", "left").join(pth, "id", "left")
             .join(cyc, "id", "left").join(fraud, "id", "left").join(dense, "id", "left")
             .na.fill({"n_shared_mule_motif": 0, "max_shared_mule_fanin": 0, "ring_co_feeders": 0,
                       "n_shared_atm": 0, "on_3cycle": 0, "n_3cycle_as_a": 0, "fraudar_score": 0.0,
                       "in_dense_block": 0, "block_density": 0.0, "passthrough_ratio": 0.0})
             .withColumn("anchor_month", F.lit(a)).persist(StorageLevel.DISK_ONLY))

    acc = feats.select(*MOTIF_COLS)
    if first_acc:
        save_overwrite(acc, OUT_MOTIF); first_acc = False
    else:
        acc.write.mode("append").saveAsTable(OUT_MOTIF)
    print("   saved account motif features for", a, "->", OUT_MOTIF)

    own = spark.table(OWN_TBL).select(F.col("dst").alias("id"), F.col("src").alias("cust"))
    cust = (feats.join(own, "id", "inner").groupBy("cust").agg(
                F.max("n_shared_mule_motif").alias("n_shared_mule_motif"),
                F.max("max_shared_mule_fanin").alias("max_shared_mule_fanin"),
                F.max("ring_co_feeders").alias("ring_co_feeders"),
                F.max("n_shared_atm").alias("n_shared_atm"),
                F.max("on_3cycle").alias("on_3cycle"),
                F.max("n_3cycle_as_a").alias("n_3cycle_as_a"),
                F.max("fraudar_score").alias("fraudar_score"),
                F.max("in_dense_block").alias("in_dense_block"),
                F.max("block_density").alias("block_density"),
                F.max("passthrough_ratio").alias("passthrough_ratio"))
            .withColumn("anchor_month", F.lit(a)))
    if first_cust:
        save_overwrite(cust, OUT_MOTIF_CUST); first_cust = False
    else:
        cust.write.mode("append").saveAsTable(OUT_MOTIF_CUST)
    print("   saved customer motif rollup for", a, "->", OUT_MOTIF_CUST)

    feats.unpersist(); e.unpersist(); rare.unpersist()
    if tri is not None: tri.unpersist()


In [ ]:
MOTIF_CUST_COLS = ["n_shared_mule_motif", "max_shared_mule_fanin", "ring_co_feeders",
                   "n_shared_atm", "on_3cycle", "n_3cycle_as_a", "fraudar_score",
                   "in_dense_block", "block_density"]
feats04 = spark.table(FEATS_CUST).where(F.col("anchor_month").isin(*anchors))
motif_c = spark.table(OUT_MOTIF_CUST)
full = (feats04.join(motif_c, ["cust", "anchor_month"], "left")
        .na.fill({c: 0 for c in MOTIF_CUST_COLS}).na.fill({"fraudar_score": 0.0, "passthrough_ratio": 0.0})
        .persist(StorageLevel.DISK_ONLY))
save_overwrite(full, OUT_MOTIF_CUST_FULL)
print("saved 04-features + motifs (customer level):", OUT_MOTIF_CUST_FULL, "| rows:", full.count())

INCREMENTS = [("04_core",     ["pr_n_shared_mule", "pr_closeness_to_bad"]),
              ("+sharedatm",  ["pr_n_shared_atm"]),
              ("+ring",       ["pr_ring_co_feeders"]),
              ("+cycles",     ["pr_on_3cycle"]),
              ("+layering",   ["pr_passthrough_ratio"]),
              ("+fraudar",    ["pr_fraudar_score"]),
              ("+densefraud", ["pr_block_density"])]
RANK_COLS = ["n_shared_mule", "closeness_to_bad", "n_shared_atm", "ring_co_feeders",
             "on_3cycle", "passthrough_ratio", "fraudar_score", "block_density"]

lift_rows = []
for a in sorted([r[0] for r in full.select("anchor_month").distinct().collect()]):
    fa = full.where(F.col("anchor_month") == a)
    for c in RANK_COLS:
        fa = pr(fa, c)
    included = []
    for step, cols in INCREMENTS:
        included = included + cols
        ens = fa.withColumn("ens", sum(F.col(c) for c in included))
        rows, n, nbad = capture_at(ens.select("target_fraud", "ens"), "ens", K_PCTS)
        for (p, K, topbad, cap, lift) in rows:
            lift_rows.append((a, step, p, K, n, nbad, topbad, cap, lift))
            print("   [%s] %-11s top%4.0f%% K=%d  capture=%.3f  lift=%.2f  (bad-cust %d/%d)"
                  % (a, step, p * 100, K, cap, lift, topbad, nbad))

lift_df = spark.createDataFrame(
    lift_rows, ["anchor_month", "step", "top_pct", "K", "n_cust", "n_bad_cust", "bad_in_topK", "capture", "lift"])
save_overwrite(lift_df, OUT_MOTIF_LIFT)
print("saved motif incremental-lift log:", OUT_MOTIF_LIFT)


In [ ]:
SLOPING_FEATURES = ["n_shared_atm", "ring_co_feeders", "max_shared_mule_fanin", "on_3cycle",
                    "n_3cycle_as_a", "fraudar_score", "in_dense_block", "block_density", "n_shared_mule_motif"]
pdf = spark.table(OUT_MOTIF_CUST_FULL).select("anchor_month", "target_fraud", *SLOPING_FEATURES).toPandas()
slope_rows = []
for a in sorted(pdf["anchor_month"].unique()):
    d = pdf[pdf["anchor_month"] == a]; y = d["target_fraud"].astype(int).values; tot_bad = int(y.sum())
    print("\n=== motif sloping @ %s (bad=%d / n=%d) ===" % (a, tot_bad, len(d)))
    for f in SLOPING_FEATURES:
        try:
            b = _robust_bins(d[f].astype(float).values, max_bins=10)
            g = pd.DataFrame({"b": b, "y": y}).groupby("b").agg(cnt=("y", "size"), bad=("y", "sum")).sort_index()
            g["event_rate_pct"] = 100.0 * g["bad"] / g["cnt"]
            g["pct_fraud_captured"] = (100.0 * g["bad"] / tot_bad) if tot_bad else 0.0
            iv = _woe_iv(g["cnt"], g["bad"]); er = g["event_rate_pct"].values
            mono = bool(np.all(np.diff(er) >= -1e-9) or np.all(np.diff(er) <= 1e-9))
            print("  %-22s IV=%.3f  bins=%d  monotonic=%s  top-bin event=%.2f%%"
                  % (f, iv, len(g), mono, er[-1] if len(er) else 0))
            for bi, r in g.reset_index().iterrows():
                slope_rows.append((a, f, int(r["b"]), int(r["cnt"]), int(r["bad"]),
                                   round(float(r["event_rate_pct"]), 4), round(float(r["pct_fraud_captured"]), 4),
                                   round(iv, 4), mono))
        except Exception as fe:
            print("  %-22s sloping skipped: %s" % (f, fe))
if slope_rows:
    sdf = spark.createDataFrame(slope_rows, ["anchor_month", "feature", "bin", "cnt", "bad",
                                             "event_rate_pct", "pct_fraud_captured", "iv", "monotonic"])
    save_overwrite(sdf, OUT_MOTIF_SLOPE)
    print("\nsaved motif sloping table:", OUT_MOTIF_SLOPE)


In [ ]:
print("=== Notebook 05 (v2) summary ===")
print("account motif features:", OUT_MOTIF, "| rows:", spark.table(OUT_MOTIF).count())
print("customer motif rollup :", OUT_MOTIF_CUST, "| rows:", spark.table(OUT_MOTIF_CUST).count())
print("Motifs: shared-beneficiary ring (ring_co_feeders, max_shared_mule_fanin), ATM cash-out (n_shared_atm),")
print("        layering 3-cycle (on_3cycle, n_3cycle_as_a), Fraudar-inspired density (fraudar_score),")
print("        Distributed Fraudar-style batch peeling for a dense subgraph (in_dense_block, block_density).")
print("Lift log:", OUT_MOTIF_LIFT, "(each motif's MARGINAL capture@K over 04's n_shared_mule+closeness).")
print("Sloping :", OUT_MOTIF_SLOPE, "(event rate, % captured, IV, monotonicity per motif).")
print("NOTEBOOK 05 (v2) COMPLETE. Keep only motifs that ADD lift over 04 -> carry validated candidates into 06 (select + score).")


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import IPython.display as _disp
from pyspark.sql import functions as _F

CUST_TBL      = f"{WRITE_DB}.{EMP}_v2_motif_cust_full"
PLOT_FEATURES = ["n_shared_atm", "ring_co_feeders", "max_shared_mule_fanin", "on_3cycle", "fraudar_score", "in_dense_block", "block_density"]
PLOT_ANCHOR   = "202401"
MIN_BIN_FRAC = float(os.environ.get("FRAUD_GRAPH_MIN_BIN_FRACTION", "0.02"))
MIN_STABLE_CNT = int(os.environ.get("FRAUD_GRAPH_MIN_STABLE_COUNT", "30"))
MIN_STABLE_BAD = int(os.environ.get("FRAUD_GRAPH_MIN_STABLE_POSITIVES", "3"))
RISKY_ON_LEFT = True

def _wilson_lcb(bad, cnt, z=1.96):
    bad = np.asarray(bad, float); cnt = np.asarray(cnt, float)
    p = np.where(cnt > 0, bad / cnt, 0.0); denom = 1 + z*z/np.maximum(cnt, 1)
    centre = p + z*z/(2*np.maximum(cnt, 1)); margin = z*np.sqrt((p*(1-p) + z*z/(4*np.maximum(cnt,1)))/np.maximum(cnt,1))
    return np.where(cnt > 0, np.clip((centre - margin)/denom, 0, 1), 0.0)

def _robust_bins(x, max_bins=10):
    s = pd.Series(np.asarray(x, dtype=float)); nuniq = int(s.nunique(dropna=True))
    if nuniq <= 1: return np.zeros(len(s), dtype=int)
    if nuniq <= max_bins:
        order = {v: i for i, v in enumerate(sorted(s.dropna().unique()))}
        return s.map(order).fillna(0).astype(int).values
    z = (s == 0); out = np.zeros(len(s), dtype=int)
    if z.mean() >= 0.02 and int((~z).sum()) >= max_bins:
        pos = ~z
        qr = pd.qcut(s[pos].rank(method="first"), q=min(max_bins - 1, int((~z).sum())), labels=False, duplicates="drop")
        out[pos.values] = 1 + qr.astype(int).values; return out
    return pd.qcut(s.rank(method="first"), q=max_bins, labels=False, duplicates="drop").astype(int).values

def _club_sparse(d, n, min_frac):
    d = d.sort_index().copy()
    while len(d) > 2:
        share = d["cnt"] / n
        if share.min() >= min_frac: break
        j = int(share.values.argmin()); k = j + 1 if j == 0 else j - 1
        lo, hi = min(j, k), max(j, k)
        d.loc[d.index[lo], ["cnt", "bad"]] += d.loc[d.index[hi], ["cnt", "bad"]].values
        d.loc[d.index[lo], "hi"] = max(d.loc[d.index[lo], "hi"], d.loc[d.index[hi], "hi"])
        d.loc[d.index[lo], "lo"] = min(d.loc[d.index[lo], "lo"], d.loc[d.index[hi], "lo"])
        d = d.drop(d.index[hi]).reset_index(drop=True)
    return d

def _woe_iv(cnt, bad):
    good = cnt - bad; tb, tg = bad.sum(), good.sum()
    pb = (bad / tb).clip(lower=1e-6); pg = (good / tg).clip(lower=1e-6)
    return float(((pg - pb) * np.log(pg / pb)).sum())

_pdf = spark.table(CUST_TBL).where(_F.col("anchor_month") == PLOT_ANCHOR).select("target_fraud", *PLOT_FEATURES).toPandas()
_y = _pdf["target_fraud"].astype(int).values; _tot_bad = int(_y.sum()); _tot_good = len(_y) - _tot_bad
print("plotting anchor %s | customers=%d | frauds=%d" % (PLOT_ANCHOR, len(_pdf), _tot_bad))

def plot_sloping(feature):
    x = _pdf[feature].astype(float).values
    d = (pd.DataFrame({"b": _robust_bins(x), "x": x, "y": _y}).groupby("b")
         .agg(cnt=("y", "size"), bad=("y", "sum"), lo=("x", "min"), hi=("x", "max")).sort_index())
    d = _club_sparse(d, len(_pdf), MIN_BIN_FRAC)
    d = (d.iloc[::-1] if RISKY_ON_LEFT else d).reset_index(drop=True)
    d["event_rate_pct"] = 100.0 * d["bad"] / d["cnt"]
    cum_bad = d["bad"].cumsum(); cum_good = (d["cnt"] - d["bad"]).cumsum()
    d["pct_captured"] = 100.0 * cum_bad / max(_tot_bad, 1)
    ks = float((100.0 * cum_bad / max(_tot_bad, 1) - 100.0 * cum_good / max(_tot_good, 1)).abs().max())
    iv = _woe_iv(d["cnt"], d["bad"])
    d["er_lcb_pct"] = 100.0 * _wilson_lcb(d["bad"].values, d["cnt"].values)
    d["stable"] = (d["cnt"] >= MIN_STABLE_CNT) & (d["bad"] >= MIN_STABLE_BAD)
    labels = [("%.3g" % lo) if lo == hi else ("[%.3g, %.3g]" % (lo, hi)) for lo, hi in zip(d["lo"], d["hi"])]
    xs = range(len(d))
    fig, ax1 = plt.subplots(figsize=(11, 5))
    ax1.bar(xs, d["event_rate_pct"], color="#c97b84", alpha=0.85, label="Event Rate (%)")
    for i, v in enumerate(d["event_rate_pct"]): ax1.text(i, v, "%.2f" % v, ha="center", va="bottom", fontsize=8)
    ax1.set_ylabel("Event Rate (%)"); ax1.set_xticks(list(xs)); ax1.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    for i, st in enumerate(d["stable"].values):
        if not st: ax1.patches[i].set_hatch("///"); ax1.patches[i].set_edgecolor("#7a2a34")
    ax2 = ax1.twinx()
    ax2.plot(xs, d["pct_captured"], color="#a23b4e", marker="o", label="Cumulative % Fraud Captured")
    ax2.set_ylabel("Cumulative % Fraud Captured"); ax2.set_ylim(0, 105)
    ax1.set_title("%s @ %s   (IV=%.3f, KS=%.1f)   hatched = UNSTABLE (n<%d)\nEvent Rate (bars) & Cumulative %% Fraud Captured (line)"
                  % (feature, PLOT_ANCHOR, iv, ks, MIN_STABLE_CNT), fontsize=12)
    ax1.legend(loc="upper left"); ax2.legend(loc="lower right"); fig.tight_layout(); plt.show()
    _disp.display(pd.DataFrame({"bin": labels, "# frauds": d["bad"].astype(int).values,
                                "# customers": d["cnt"].astype(int).values,
                                "event_rate_%": d["event_rate_pct"].round(3).values,
                                "event_rate_LCB_%": d["er_lcb_pct"].round(2).values,
                                "cum_%_captured": d["pct_captured"].round(2).values,
                                "stable": d["stable"].values}))

for _f in PLOT_FEATURES:
    try: plot_sloping(_f)
    except Exception as _e: print("  %s: skipped (%s)" % (_f, _e))
